# p4-onset RL — Kaggle 远程 PPO Worker

连接到本地 hub 的 cloudflared tunnel，拉取 p4-onset 课程 job 执行 PPO 更新。

## 工作原理

| 组件 | 位置 | 职责 |
|---|---|---|
| **hub** | 本地 Windows 训练机 | 采集中枢 + Rollout 派发（LAN 节点）+ 训练主循环 + cloudflared tunnel |
| **LAN 节点** | 本地局域网机器 | 只收 rollout 任务、跑局、回报（零改动） |
| **Kaggle (本 notebook)** | 云端 GPU | 无状态 PPO worker：轮询 hub → 下载 payload → PPO 更新 → 回传权重 |

课程配置为 **p4-onset**（四面围攻）：4 敌混编（basic/fast/power/armor），
玩家居中，1 命 0 星，hard 难度。奖励方案：杀敌 +3.0、命中 +0.3、
拾取 +1.5、挨打 -1.0、反蹲坑、弹药效率。

## 使用前

1. 确保 hub 端已启动：`hub-server` + `cloudflared tunnel` + `training loop`
2. hub 端 `rl-config.json` 中 `rl.per-tick.course` 指向 `p4-onset.jsonc`
3. 在下方的 `⚙️ 参数配置` 单元格填入当前 tunnel URL 和 token
4. 依次运行各单元格

---
## ⚙️ 参数配置

In [ ]:
# @title 填入 hub 连接信息
import os
import sys
import time

# 手动填入以下参数
# HUB_URL 和 HUB_TOKEN 由 hub 端提供，见 hub 的 rl-config.json 中 remote_hub_url / remote_token
HUB_URL = "https://your-tunnel.trycloudflare.com"
HUB_TOKEN = "YOUR_TOKEN_HERE"

# 保活配置：Kaggle GPU 会话最长 9h，30h/周配额
MAX_SESSION_HOURS = 9
POLL_INTERVAL_SEC = 5

print(f"[{time.strftime('%H:%M:%S')}] HUB_URL = '{HUB_URL}'")
print(f"[{time.strftime('%H:%M:%S')}] HUB_TOKEN len = {len(HUB_TOKEN)}")
print(f"[{time.strftime('%H:%M:%S')}] Platform = Kaggle")
print(f"[{time.strftime('%H:%M:%S')}] Max session = {MAX_SESSION_HOURS}h")

---
## 1. 安装依赖

Kaggle 默认环境已预装 PyTorch（CUDA 版），仅需确认版本并按需补齐。

In [ ]:
import subprocess

import torch


def run(cmd, **kw):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, **kw)


# 确认 torch 可用（Kaggle 通常预装 torch+cu118+）
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[{time.strftime('%H:%M:%S')}] torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[{time.strftime('%H:%M:%S')}]   device: {torch.cuda.get_device_name(0)}")
    print(f"[{time.strftime('%H:%M:%S')}]   CUDA capability: {torch.cuda.get_device_capability()}")

# 若 torch 版本过旧，可升级（Kaggle 2026 环境通常自带 >=2.0）
if int(torch.__version__.split('.')[0]) < 2:
    print(f"[{time.strftime('%H:%M:%S')}] torch 版本过旧，升级中...")
    run(f"{sys.executable} -m pip install --quiet --upgrade torch")

print(f"[{time.strftime('%H:%M:%S')}] Dependencies ready")

---
## 2. Kaggle 会话保活

Kaggle GPU 会话最长 9 小时，每周 GPU 配额 30 小时。
通过定期输出防止空闲超时（Kaggle 在 notebook 有输出时不回收）。
作业完成后 notebook 会自动退出，不浪费配额。

In [ ]:
import threading

KEEPALIVE_STOP = threading.Event()


def keepalive_loop():
    """每 120s 打印一次心跳，防止 Kaggle 空闲回收。"""
    n = 0
    while not KEEPALIVE_STOP.is_set():
        print(f"[{time.strftime('%H:%M:%S')}] [keepalive] alive ({n * 2} min elapsed)")
        n += 1
        KEEPALIVE_STOP.wait(120)
    print(f"[{time.strftime('%H:%M:%S')}] [keepalive] stopped after {n} pings ({n * 2} min)")


th = threading.Thread(target=keepalive_loop, daemon=True, name="kaggle-keepalive")
th.start()
print(f"[{time.strftime('%H:%M:%S')}] Keepalive thread started (every 120s, max {MAX_SESSION_HOURS}h)")
print(f"[{time.strftime('%H:%M:%S')}] INFO: Kaggle 9h GPU session limit — worker will auto-exit via --max-idle-sec")

---
## 3. 下载 code.zip 并启动远程 PPO Worker

从 hub 下载 `code.zip`（hub 启动时打包的代码快照）→ 解压到 `sys.path` → 启动 worker 轮询。
代码一致性由 `code_sha256` 保证，无需 git clone。

> 可以随时中断此单元格（`Kernel → Interrupt`），worker 会优雅退出。
> 中断后如还有配额，可重新运行此单元格继续轮询。

### 关于 p4-onset 课程

课程配置在 hub 端 `nn-training/curricula/p4-onset.jsonc`，由 `rl-config.json` 的
`rl.per-tick.course` 指针指定。本 worker 不关心具体课程内容——hub 将课程快照全文
写入 job manifest，worker 按 manifest 中的 reward 公式和超参执行 PPO。

p4-onset 关键参数（由 hub 端课程配置决定，worker 仅读取 manifest）：
- **PPO 超参**：epochs=4, mb=512, lr=0.00015, gamma=0.995, lam=0.97
- **PPO schedule**：kl_coef 从 0.6 衰减到 0.0，lr 同步衰减
- **奖励**：杀敌 +3.0, 命中 +0.3, 拾取 +1.5, 挨打 -1.0, 反蹲坑 -0.02/tick, 弹药效率 -0.01/发
- **评估**：每 5 轮评估 20 局/关

> 注意：远程模式每迭代结算一次 PPO（非 wave 级 stream），
> 云 worker 跑完一轮 PPO 后回传权重，hub 校验后落盘并继续下一轮。

In [ ]:
import io
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

WORK_DIR = Path("/tmp/remote-worker")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"[{time.strftime('%H:%M:%S')}] [kaggle] Downloading code.zip from {HUB_URL}/code...")
req = urllib.request.Request(
    f"{HUB_URL.rstrip('/')}/code",
    headers={"Authorization": f"Bearer {HUB_TOKEN}"},
)
try:
    with urllib.request.urlopen(req, timeout=120) as resp:
        code_raw = resp.read()
        print(f"[{time.strftime('%H:%M:%S')}] [kaggle] code.zip: {len(code_raw)} bytes (HTTP {resp.status})")
except urllib.error.HTTPError as e:
    print(f"[{time.strftime('%H:%M:%S')}] [kaggle] FAILED to download code.zip: HTTP {e.code} — {e.read().decode()[:200]}")
    raise SystemExit(1) from None
except Exception as e:
    print(f"[{time.strftime('%H:%M:%S')}] [kaggle] FAILED to connect to hub: {e}")
    print(f"[{time.strftime('%H:%M:%S')}] [kaggle] Check HUB_URL and try again")
    raise SystemExit(1) from None

# 解压到 sys.path，后续 import remote.worker 从 code.zip 加载
CODE_DIR = Path("/tmp/worker-code")
CODE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(code_raw)) as z:
    z.extractall(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
print(f"[{time.strftime('%H:%M:%S')}] [kaggle] code.zip extracted -> {CODE_DIR} (sys.path[0])")

from remote.worker import worker_loop

print(f"[{time.strftime('%H:%M:%S')}] [kaggle] Testing connectivity to hub...")
req = urllib.request.Request(
    f"{HUB_URL.rstrip('/')}/ping",
    headers={"Authorization": f"Bearer {HUB_TOKEN}"},
)
try:
    with urllib.request.urlopen(req, timeout=15) as resp:
        print(f"[{time.strftime('%H:%M:%S')}] [kaggle] hub HTTP {resp.status}")
except Exception as e:
    print(f"[{time.strftime('%H:%M:%S')}] [kaggle] CONNECTION FAILED: {e}")
    print(f"[{time.strftime('%H:%M:%S')}] [kaggle] Check HUB_URL and try again")
    raise SystemExit(1) from None

print(f"[{time.strftime('%H:%M:%S')}] [kaggle] Starting worker loop...")
# 最大空闲时间 = MAX_SESSION_HOURS - 1h 缓冲（保活线程在后台运行）
max_idle = max(3600, (MAX_SESSION_HOURS - 1) * 3600)
t_start = time.time()

try:
    n = worker_loop(
        HUB_URL,
        HUB_TOKEN,
        work_dir=WORK_DIR,
        device=DEVICE,
        torch_threads=0,
        poll_sec=POLL_INTERVAL_SEC,
        once=False,
        max_idle_sec=max_idle,
    )
    print(f"\n[{time.strftime('%H:%M:%S')}] [kaggle] Worker exited: {n} job(s) processed")
except KeyboardInterrupt:
    print(f"\n[{time.strftime('%H:%M:%S')}] [kaggle] Worker interrupted by user")
finally:
    KEEPALIVE_STOP.set()

elapsed = time.time() - t_start
print(f"[{time.strftime('%H:%M:%S')}] [kaggle] Session duration: {elapsed/60:.1f} min")

---
## 4. 停止保活

如果提前中断了 worker，运行此单元格停止保活线程。

In [ ]:
KEEPALIVE_STOP.set()
print(f"[{time.strftime('%H:%M:%S')}] Keepalive stopped")

---
## 5. 会话管理与故障排查

### Kaggle GPU 配额管理

- **每周 30h GPU 配额**：Kaggle 每周一 UTC 重置，请合理分配
- **单次 9h 上限**：会话最长 9h，worker 会在空闲超时后自动退出
- **中断后重连**：Kaggle 允许在同一个 notebook 中多次运行 cell——
  hub 的 job 幂等机制保证不会重复训练同一轮

### 预计墙钟

| 阶段 | 预估时间 | 说明 |
|---|---|---|
| 安装依赖 | ~30s | torch 通常已预装 |
| 下载 code.zip | ~5s | ~1-2 MB hub 代码快照 |
| 下载 payload | ~10-30s | 单轮 ~18.5 MB zip（140× 压缩比） |
| PPO 更新（T4） | ~30s-2min | epochs=4, mb=512 约 1min |
| 回传结果 | ~2-5s | weights_json + opt tar < 5 MB |

### 预期日志

hub 端：
```
[hub-server] "GET /jobs/next HTTP/1.1" 200 -
[hub-server] "GET /jobs/{id}/payload HTTP/1.1" 200 -
[hub-server] "POST /jobs/{id}/result HTTP/1.1" 200 -
```

worker 端：
```
[worker] job {id} claimed — downloading payload
[worker] job {id}: code.zip unpacked (N bytes, M .py files) -> sys.path[0]
[worker] job {id}: PPO done in {sec}s, steps={n} chunks={m} kl={k}
[worker] job {id} done — result accepted
```

### 常见问题

| 症状 | 原因 | 处理 |
|---|---|---|
| `CONNECTION FAILED` | hub tunnel 未启动或 URL 过期 | 确认 hub 端 tunnel 运行中，更新 HUB_URL |
| `HTTP 401` | token 不匹配 | 检查 HUB_TOKEN 与 hub 端一致 |
| `payload_sha256 不匹配` | 传输损坏 | 自动重试，hub 的 job 幂等机制保证不重复 |
| `commit 不一致` | hub 代码已更新但未 push | 通知 hub 端 push 当前分支 |
| `D14 course_fp 不匹配` | 课程切换导致语料血缘变更 | 正常现象，hub 下轮发布新课程 job |
| 长时间无 job | rollout 采集未完成 | hub 端 rollout 采集完成后自动发布 job |
| Kaggle 配额耗尽 | 30h/周 GPU 用完 | 等待周一重置，或使用 Colab Pro/AutoDL 替代 |

---
## 附录：p4-onset 课程摘要

```jsonc
// nn-training/curricula/p4-onset.jsonc（hub 端配置）
{
  "name": "p4-onset",
  "mode": "per-tick",
  "stages": [{
    "name": "p4-onset-mixed",
    "grid": 13×13 cells（空旷场 + 围墙边界）,
    "forces": "abcdabcdabcdabcdabcd",  // 4 敌混编
    "player_spawn": { "col": 12, "row": 12 },  // 正中
    "enemy_spawns": [四角: TL/TR/BR/BL]
  }],
  "difficulty": "hard",
  "max_ticks": 2400,
  "player": { "lives": 1, "level": 0 },
  "reward": {
    "formula": "wKill*kills + wHit*enemyHits + wPickup*powerUpsCollected + ...",
    "params": { "wKill": 3.0, "wHit": 0.3, "wDmg": 1.0, ... }
  },
  "epochs": 4, "mb": 512, "lr": 0.00015,
  "gamma": 0.995, "lam": 0.97,
  "ppo_schedule": [
    { "until_iter": 15, "kl_coef": 0.6, "lr": 0.0003 },
    { "until_iter": 35, "kl_coef": 0.2, "lr": 0.00015 },
    { "kl_coef": 0.0, "lr": 0.00005 }
  ]
}
```

行为基线（God-AI vs p1-ep60 起点）：
- God-AI: **64/100** 胜率（杀 306，均 3.06）
- p1-ep60（RL 起点）: **14/100** 胜率（杀 193）—— 泛化缺口 ~50pp

RL 目标：从 p1-ep60 权重起步，在 p4-onset 上通过 PPO 迭代将胜率提升至 God-AI 水平（~64/100）以上。